### 1. Import Liabraries

In [1]:
import pandas as pd
import numpy as np
import re

### 2. Load raw CSV File

In [2]:
df = pd.read_csv("CarDekho_India_UsedCars.csv")
df.head()

,City,Title,Details,Price,Old Price,Savings,Image,Link
0,delhi,2024 Kia Sonet HTK Plus,"10,000 kms • Petrol • Manual",₹8.40 Lakh,NaN,NaN,https://images10.gaadi.com/usedcar_image/48612...,https://www.cardekho.com/used-car-details/used...
1,delhi,2022 Renault Kiger RXZ,"50,214 kms • Petrol • Manual",₹5.75 Lakh,NaN,NaN,https://images10.gaadi.com/usedcar_image/49351...,https://www.cardekho.com/buy-used-car-details/...
2,delhi,2024 Nissan Magnite XV,"19,000 kms • Petrol • Manual",₹6.80 Lakh,NaN,NaN,https://images10.gaadi.com/usedcar_image/49251...,https://www.cardekho.com/used-car-details/used...
3,delhi,2022 Renault Kiger RXT Opt,"14,464 kms • Petrol • Manual",₹5.50 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/buy-used-car-details/...
4,delhi,2022 Kia Sonet HTX Turbo iMT BSVI,"37,741 kms • Petrol • Manual",₹8 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...


### 3. Clean Details - KM, Fuel, Transmission

In [3]:
df[['KM_Driven', 'Fuel_Type', 'Transmission']] = df['Details'].str.split('•', expand=True)

df['KM_Driven'] = (
    df['KM_Driven'].str.replace("kms", "", regex=False)
                   .str.replace(",", "")
                   .str.strip()
)

df['KM_Driven'] = pd.to_numeric(df['KM_Driven'], errors='coerce')

df['Fuel_Type'] = df['Fuel_Type'].str.strip()
df['Transmission'] = df['Transmission'].str.strip()

In [4]:
df

,City,Title,Details,Price,Old Price,Savings,Image,Link,KM_Driven,Fuel_Type,Transmission
0,delhi,2024 Kia Sonet HTK Plus,"10,000 kms • Petrol • Manual",₹8.40 Lakh,NaN,NaN,https://images10.gaadi.com/usedcar_image/48612...,https://www.cardekho.com/used-car-details/used...,10000,Petrol,Manual
1,delhi,2022 Renault Kiger RXZ,"50,214 kms • Petrol • Manual",₹5.75 Lakh,NaN,NaN,https://images10.gaadi.com/usedcar_image/49351...,https://www.cardekho.com/buy-used-car-details/...,50214,Petrol,Manual
2,delhi,2024 Nissan Magnite XV,"19,000 kms • Petrol • Manual",₹6.80 Lakh,NaN,NaN,https://images10.gaadi.com/usedcar_image/49251...,https://www.cardekho.com/used-car-details/used...,19000,Petrol,Manual
3,delhi,2022 Renault Kiger RXT Opt,"14,464 kms • Petrol • Manual",₹5.50 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/buy-used-car-details/...,14464,Petrol,Manual
4,delhi,2022 Kia Sonet HTX Turbo iMT BSVI,"37,741 kms • Petrol • Manual",₹8 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...,37741,Petrol,Manual
...,...,...,...,...,...,...,...,...,...,...,...
15595,chandigarh,2023 Maruti Alto K10 VXI,"28,199 kms • Petrol • Manual",₹3.80 Lakh,₹4.19 Lakh,"(Save ₹39,131)",https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/buy-used-car-details/...,28199,Petrol,Manual
15596,chandigarh,2024 MG Astor Select CVT,"20,000 kms • Petrol • Automatic",₹12.50 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...,20000,Petrol,Automatic
15597,chandigarh,2024 Mahindra XUV 3XO MX3,"8,000 kms • Petrol • Manual",₹8.50 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...,8000,Petrol,Manual
15598,chandigarh,2024 Kia Sonet HTX Turbo iMT,"17,000 kms • Petrol • Manual",₹10.70 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...,17000,Petrol,Manual


### 4. Converts Price values to Lakhs

In [5]:
def to_lakh(x):
    if pd.isna(x):
        return None
    
    x = str(x).replace("₹","").replace(",","").strip().lower()
    
    # exponential wrong values: divide by 100000 twice
    if "e+" in x:
        val = float(x)
        return round(val / 100000 / 100000, 2)
    
    # values written in lakhs
    if any(k in x for k in ["lakh", "lac", "lacs"]):
        num = re.findall(r"[0-9]*\.?[0-9]+", x)
        return round(float(num[0]), 2)
    
    # direct rupee values
    if x.isdigit():
        return round(float(x) / 100000, 2)
    
    # last fallback
    nums = re.findall(r"[0-9]*\.?[0-9]+", x)
    if nums:
        return round(float(nums[0]) / 100000, 2)
    
    return None

df["Price_Lakh"] = df["Price"].apply(to_lakh)

In [6]:
df

,City,Title,Details,Price,Old Price,Savings,Image,Link,KM_Driven,Fuel_Type,Transmission,Price_Lakh
0,delhi,2024 Kia Sonet HTK Plus,"10,000 kms • Petrol • Manual",₹8.40 Lakh,NaN,NaN,https://images10.gaadi.com/usedcar_image/48612...,https://www.cardekho.com/used-car-details/used...,10000,Petrol,Manual,8.40
1,delhi,2022 Renault Kiger RXZ,"50,214 kms • Petrol • Manual",₹5.75 Lakh,NaN,NaN,https://images10.gaadi.com/usedcar_image/49351...,https://www.cardekho.com/buy-used-car-details/...,50214,Petrol,Manual,5.75
2,delhi,2024 Nissan Magnite XV,"19,000 kms • Petrol • Manual",₹6.80 Lakh,NaN,NaN,https://images10.gaadi.com/usedcar_image/49251...,https://www.cardekho.com/used-car-details/used...,19000,Petrol,Manual,6.80
3,delhi,2022 Renault Kiger RXT Opt,"14,464 kms • Petrol • Manual",₹5.50 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/buy-used-car-details/...,14464,Petrol,Manual,5.50
4,delhi,2022 Kia Sonet HTX Turbo iMT BSVI,"37,741 kms • Petrol • Manual",₹8 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...,37741,Petrol,Manual,8.00
...,...,...,...,...,...,...,...,...,...,...,...,...
15595,chandigarh,2023 Maruti Alto K10 VXI,"28,199 kms • Petrol • Manual",₹3.80 Lakh,₹4.19 Lakh,"(Save ₹39,131)",https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/buy-used-car-details/...,28199,Petrol,Manual,3.80
15596,chandigarh,2024 MG Astor Select CVT,"20,000 kms • Petrol • Automatic",₹12.50 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...,20000,Petrol,Automatic,12.50
15597,chandigarh,2024 Mahindra XUV 3XO MX3,"8,000 kms • Petrol • Manual",₹8.50 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...,8000,Petrol,Manual,8.50
15598,chandigarh,2024 Kia Sonet HTX Turbo iMT,"17,000 kms • Petrol • Manual",₹10.70 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...,17000,Petrol,Manual,10.70


### 5. Convert Old Price and Savings into Lakhs

In [7]:
df["OldPrice_Lakh"] = df["Old Price"].apply(to_lakh)
df["Savings_Lakh"]  = df["Savings"].apply(to_lakh)

In [8]:
df

,City,Title,Details,Price,Old Price,Savings,Image,Link,KM_Driven,Fuel_Type,Transmission,Price_Lakh,OldPrice_Lakh,Savings_Lakh
0,delhi,2024 Kia Sonet HTK Plus,"10,000 kms • Petrol • Manual",₹8.40 Lakh,NaN,NaN,https://images10.gaadi.com/usedcar_image/48612...,https://www.cardekho.com/used-car-details/used...,10000,Petrol,Manual,8.40,NaN,NaN
1,delhi,2022 Renault Kiger RXZ,"50,214 kms • Petrol • Manual",₹5.75 Lakh,NaN,NaN,https://images10.gaadi.com/usedcar_image/49351...,https://www.cardekho.com/buy-used-car-details/...,50214,Petrol,Manual,5.75,NaN,NaN
2,delhi,2024 Nissan Magnite XV,"19,000 kms • Petrol • Manual",₹6.80 Lakh,NaN,NaN,https://images10.gaadi.com/usedcar_image/49251...,https://www.cardekho.com/used-car-details/used...,19000,Petrol,Manual,6.80,NaN,NaN
3,delhi,2022 Renault Kiger RXT Opt,"14,464 kms • Petrol • Manual",₹5.50 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/buy-used-car-details/...,14464,Petrol,Manual,5.50,NaN,NaN
4,delhi,2022 Kia Sonet HTX Turbo iMT BSVI,"37,741 kms • Petrol • Manual",₹8 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...,37741,Petrol,Manual,8.00,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15595,chandigarh,2023 Maruti Alto K10 VXI,"28,199 kms • Petrol • Manual",₹3.80 Lakh,₹4.19 Lakh,"(Save ₹39,131)",https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/buy-used-car-details/...,28199,Petrol,Manual,3.80,4.19,0.39
15596,chandigarh,2024 MG Astor Select CVT,"20,000 kms • Petrol • Automatic",₹12.50 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...,20000,Petrol,Automatic,12.50,NaN,NaN
15597,chandigarh,2024 Mahindra XUV 3XO MX3,"8,000 kms • Petrol • Manual",₹8.50 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...,8000,Petrol,Manual,8.50,NaN,NaN
15598,chandigarh,2024 Kia Sonet HTX Turbo iMT,"17,000 kms • Petrol • Manual",₹10.70 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...,17000,Petrol,Manual,10.70,NaN,NaN


### 6. Extract Year, Brand, Model

In [9]:
df["Year"] = df["Title"].str.extract(r'(\d{4})').astype(int)

df["Brand"] = df["Title"].str.split().str[1]

df["Model"] = df["Title"].str.replace(r'^\d{4}\s+', '', regex=True)

In [10]:
df

,City,Title,Details,Price,Old Price,Savings,Image,Link,KM_Driven,Fuel_Type,Transmission,Price_Lakh,OldPrice_Lakh,Savings_Lakh,Year,Brand,Model
0,delhi,2024 Kia Sonet HTK Plus,"10,000 kms • Petrol • Manual",₹8.40 Lakh,NaN,NaN,https://images10.gaadi.com/usedcar_image/48612...,https://www.cardekho.com/used-car-details/used...,10000,Petrol,Manual,8.40,NaN,NaN,2024,Kia,Kia Sonet HTK Plus
1,delhi,2022 Renault Kiger RXZ,"50,214 kms • Petrol • Manual",₹5.75 Lakh,NaN,NaN,https://images10.gaadi.com/usedcar_image/49351...,https://www.cardekho.com/buy-used-car-details/...,50214,Petrol,Manual,5.75,NaN,NaN,2022,Renault,Renault Kiger RXZ
2,delhi,2024 Nissan Magnite XV,"19,000 kms • Petrol • Manual",₹6.80 Lakh,NaN,NaN,https://images10.gaadi.com/usedcar_image/49251...,https://www.cardekho.com/used-car-details/used...,19000,Petrol,Manual,6.80,NaN,NaN,2024,Nissan,Nissan Magnite XV
3,delhi,2022 Renault Kiger RXT Opt,"14,464 kms • Petrol • Manual",₹5.50 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/buy-used-car-details/...,14464,Petrol,Manual,5.50,NaN,NaN,2022,Renault,Renault Kiger RXT Opt
4,delhi,2022 Kia Sonet HTX Turbo iMT BSVI,"37,741 kms • Petrol • Manual",₹8 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...,37741,Petrol,Manual,8.00,NaN,NaN,2022,Kia,Kia Sonet HTX Turbo iMT BSVI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15595,chandigarh,2023 Maruti Alto K10 VXI,"28,199 kms • Petrol • Manual",₹3.80 Lakh,₹4.19 Lakh,"(Save ₹39,131)",https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/buy-used-car-details/...,28199,Petrol,Manual,3.80,4.19,0.39,2023,Maruti,Maruti Alto K10 VXI
15596,chandigarh,2024 MG Astor Select CVT,"20,000 kms • Petrol • Automatic",₹12.50 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...,20000,Petrol,Automatic,12.50,NaN,NaN,2024,MG,MG Astor Select CVT
15597,chandigarh,2024 Mahindra XUV 3XO MX3,"8,000 kms • Petrol • Manual",₹8.50 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...,8000,Petrol,Manual,8.50,NaN,NaN,2024,Mahindra,Mahindra XUV 3XO MX3
15598,chandigarh,2024 Kia Sonet HTX Turbo iMT,"17,000 kms • Petrol • Manual",₹10.70 Lakh,NaN,NaN,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...,17000,Petrol,Manual,10.70,NaN,NaN,2024,Kia,Kia Sonet HTX Turbo iMT


### 7. Remove Unwanted Columns

In [11]:
df.drop(columns=["Details", "Price_Lakh", "OldPrice_Lakh", "Savings_Lakh", "Image", "Link" ], inplace=True)

In [12]:
df

,City,Title,Price,Old Price,Savings,KM_Driven,Fuel_Type,Transmission,Year,Brand,Model
0,delhi,2024 Kia Sonet HTK Plus,₹8.40 Lakh,NaN,NaN,10000,Petrol,Manual,2024,Kia,Kia Sonet HTK Plus
1,delhi,2022 Renault Kiger RXZ,₹5.75 Lakh,NaN,NaN,50214,Petrol,Manual,2022,Renault,Renault Kiger RXZ
2,delhi,2024 Nissan Magnite XV,₹6.80 Lakh,NaN,NaN,19000,Petrol,Manual,2024,Nissan,Nissan Magnite XV
3,delhi,2022 Renault Kiger RXT Opt,₹5.50 Lakh,NaN,NaN,14464,Petrol,Manual,2022,Renault,Renault Kiger RXT Opt
4,delhi,2022 Kia Sonet HTX Turbo iMT BSVI,₹8 Lakh,NaN,NaN,37741,Petrol,Manual,2022,Kia,Kia Sonet HTX Turbo iMT BSVI
...,...,...,...,...,...,...,...,...,...,...,...
15595,chandigarh,2023 Maruti Alto K10 VXI,₹3.80 Lakh,₹4.19 Lakh,"(Save ₹39,131)",28199,Petrol,Manual,2023,Maruti,Maruti Alto K10 VXI
15596,chandigarh,2024 MG Astor Select CVT,₹12.50 Lakh,NaN,NaN,20000,Petrol,Automatic,2024,MG,MG Astor Select CVT
15597,chandigarh,2024 Mahindra XUV 3XO MX3,₹8.50 Lakh,NaN,NaN,8000,Petrol,Manual,2024,Mahindra,Mahindra XUV 3XO MX3
15598,chandigarh,2024 Kia Sonet HTX Turbo iMT,₹10.70 Lakh,NaN,NaN,17000,Petrol,Manual,2024,Kia,Kia Sonet HTX Turbo iMT


### 8. Save Cleaned CSV file

In [13]:
df.to_csv("CarDekho_India_UsedCars_CLEANED.csv", index=False)
df

,City,Title,Price,Old Price,Savings,KM_Driven,Fuel_Type,Transmission,Year,Brand,Model
0,delhi,2024 Kia Sonet HTK Plus,₹8.40 Lakh,NaN,NaN,10000,Petrol,Manual,2024,Kia,Kia Sonet HTK Plus
1,delhi,2022 Renault Kiger RXZ,₹5.75 Lakh,NaN,NaN,50214,Petrol,Manual,2022,Renault,Renault Kiger RXZ
2,delhi,2024 Nissan Magnite XV,₹6.80 Lakh,NaN,NaN,19000,Petrol,Manual,2024,Nissan,Nissan Magnite XV
3,delhi,2022 Renault Kiger RXT Opt,₹5.50 Lakh,NaN,NaN,14464,Petrol,Manual,2022,Renault,Renault Kiger RXT Opt
4,delhi,2022 Kia Sonet HTX Turbo iMT BSVI,₹8 Lakh,NaN,NaN,37741,Petrol,Manual,2022,Kia,Kia Sonet HTX Turbo iMT BSVI
...,...,...,...,...,...,...,...,...,...,...,...
15595,chandigarh,2023 Maruti Alto K10 VXI,₹3.80 Lakh,₹4.19 Lakh,"(Save ₹39,131)",28199,Petrol,Manual,2023,Maruti,Maruti Alto K10 VXI
15596,chandigarh,2024 MG Astor Select CVT,₹12.50 Lakh,NaN,NaN,20000,Petrol,Automatic,2024,MG,MG Astor Select CVT
15597,chandigarh,2024 Mahindra XUV 3XO MX3,₹8.50 Lakh,NaN,NaN,8000,Petrol,Manual,2024,Mahindra,Mahindra XUV 3XO MX3
15598,chandigarh,2024 Kia Sonet HTX Turbo iMT,₹10.70 Lakh,NaN,NaN,17000,Petrol,Manual,2024,Kia,Kia Sonet HTX Turbo iMT


In [14]:
df.replace(["", " ", "--", "None", "none", "NULL", "null", "N/A", "nan", "NaN"], 0, inplace=True)

In [15]:
df

,City,Title,Price,Old Price,Savings,KM_Driven,Fuel_Type,Transmission,Year,Brand,Model
0,delhi,2024 Kia Sonet HTK Plus,₹8.40 Lakh,NaN,NaN,10000,Petrol,Manual,2024,Kia,Kia Sonet HTK Plus
1,delhi,2022 Renault Kiger RXZ,₹5.75 Lakh,NaN,NaN,50214,Petrol,Manual,2022,Renault,Renault Kiger RXZ
2,delhi,2024 Nissan Magnite XV,₹6.80 Lakh,NaN,NaN,19000,Petrol,Manual,2024,Nissan,Nissan Magnite XV
3,delhi,2022 Renault Kiger RXT Opt,₹5.50 Lakh,NaN,NaN,14464,Petrol,Manual,2022,Renault,Renault Kiger RXT Opt
4,delhi,2022 Kia Sonet HTX Turbo iMT BSVI,₹8 Lakh,NaN,NaN,37741,Petrol,Manual,2022,Kia,Kia Sonet HTX Turbo iMT BSVI
...,...,...,...,...,...,...,...,...,...,...,...
15595,chandigarh,2023 Maruti Alto K10 VXI,₹3.80 Lakh,₹4.19 Lakh,"(Save ₹39,131)",28199,Petrol,Manual,2023,Maruti,Maruti Alto K10 VXI
15596,chandigarh,2024 MG Astor Select CVT,₹12.50 Lakh,NaN,NaN,20000,Petrol,Automatic,2024,MG,MG Astor Select CVT
15597,chandigarh,2024 Mahindra XUV 3XO MX3,₹8.50 Lakh,NaN,NaN,8000,Petrol,Manual,2024,Mahindra,Mahindra XUV 3XO MX3
15598,chandigarh,2024 Kia Sonet HTX Turbo iMT,₹10.70 Lakh,NaN,NaN,17000,Petrol,Manual,2024,Kia,Kia Sonet HTX Turbo iMT


In [16]:
df.replace(["", " ", "--", "None", "none", "NULL", "null", "N/A", "nan", "NaN"], 0, inplace=True)
df = df.fillna(0)

In [17]:
df

,City,Title,Price,Old Price,Savings,KM_Driven,Fuel_Type,Transmission,Year,Brand,Model
0,delhi,2024 Kia Sonet HTK Plus,₹8.40 Lakh,0,0,10000,Petrol,Manual,2024,Kia,Kia Sonet HTK Plus
1,delhi,2022 Renault Kiger RXZ,₹5.75 Lakh,0,0,50214,Petrol,Manual,2022,Renault,Renault Kiger RXZ
2,delhi,2024 Nissan Magnite XV,₹6.80 Lakh,0,0,19000,Petrol,Manual,2024,Nissan,Nissan Magnite XV
3,delhi,2022 Renault Kiger RXT Opt,₹5.50 Lakh,0,0,14464,Petrol,Manual,2022,Renault,Renault Kiger RXT Opt
4,delhi,2022 Kia Sonet HTX Turbo iMT BSVI,₹8 Lakh,0,0,37741,Petrol,Manual,2022,Kia,Kia Sonet HTX Turbo iMT BSVI
...,...,...,...,...,...,...,...,...,...,...,...
15595,chandigarh,2023 Maruti Alto K10 VXI,₹3.80 Lakh,₹4.19 Lakh,"(Save ₹39,131)",28199,Petrol,Manual,2023,Maruti,Maruti Alto K10 VXI
15596,chandigarh,2024 MG Astor Select CVT,₹12.50 Lakh,0,0,20000,Petrol,Automatic,2024,MG,MG Astor Select CVT
15597,chandigarh,2024 Mahindra XUV 3XO MX3,₹8.50 Lakh,0,0,8000,Petrol,Manual,2024,Mahindra,Mahindra XUV 3XO MX3
15598,chandigarh,2024 Kia Sonet HTX Turbo iMT,₹10.70 Lakh,0,0,17000,Petrol,Manual,2024,Kia,Kia Sonet HTX Turbo iMT


In [18]:
df.to_csv("CarDekho_India_UsedCars_CLEANED.csv", index=False)

In [19]:
df

,City,Title,Price,Old Price,Savings,KM_Driven,Fuel_Type,Transmission,Year,Brand,Model
0,delhi,2024 Kia Sonet HTK Plus,₹8.40 Lakh,0,0,10000,Petrol,Manual,2024,Kia,Kia Sonet HTK Plus
1,delhi,2022 Renault Kiger RXZ,₹5.75 Lakh,0,0,50214,Petrol,Manual,2022,Renault,Renault Kiger RXZ
2,delhi,2024 Nissan Magnite XV,₹6.80 Lakh,0,0,19000,Petrol,Manual,2024,Nissan,Nissan Magnite XV
3,delhi,2022 Renault Kiger RXT Opt,₹5.50 Lakh,0,0,14464,Petrol,Manual,2022,Renault,Renault Kiger RXT Opt
4,delhi,2022 Kia Sonet HTX Turbo iMT BSVI,₹8 Lakh,0,0,37741,Petrol,Manual,2022,Kia,Kia Sonet HTX Turbo iMT BSVI
...,...,...,...,...,...,...,...,...,...,...,...
15595,chandigarh,2023 Maruti Alto K10 VXI,₹3.80 Lakh,₹4.19 Lakh,"(Save ₹39,131)",28199,Petrol,Manual,2023,Maruti,Maruti Alto K10 VXI
15596,chandigarh,2024 MG Astor Select CVT,₹12.50 Lakh,0,0,20000,Petrol,Automatic,2024,MG,MG Astor Select CVT
15597,chandigarh,2024 Mahindra XUV 3XO MX3,₹8.50 Lakh,0,0,8000,Petrol,Manual,2024,Mahindra,Mahindra XUV 3XO MX3
15598,chandigarh,2024 Kia Sonet HTX Turbo iMT,₹10.70 Lakh,0,0,17000,Petrol,Manual,2024,Kia,Kia Sonet HTX Turbo iMT


In [20]:
import pandas as pd

# Your CSV load
df = pd.read_csv("CarDekho_India_UsedCars_CLEANED.csv")

# 1) Extract Year (first 4 digits always year)
df["Year"] = df["Title"].str.extract(r"(^\d{4})")

# 2) Remove Year from title string
df["Title_temp"] = df["Title"].str.replace(r"^\d{4}\s*", "", regex=True)

# 3) Split remaining text into Brand + Model + Variant
split_cols = df["Title_temp"].str.split(" ", n=2, expand=True)

df["Brand"] = split_cols[0]         # First word = Brand
df["Model"] = split_cols[1]         # Second word = Model
df["Variant"] = split_cols[2]       # Remaining = Variant

# 4) Clean up
df.drop(columns=["Title_temp"], inplace=True)

# 5) Convert Year to integer
df["Year"] = df["Year"].astype(int)

# Save cleaned file
#df.to_csv("CarDekho_CLEANED.csv", index=False)

#df.head()


In [21]:
df

,City,Title,Price,Old Price,Savings,KM_Driven,Fuel_Type,Transmission,Year,Brand,Model,Variant
0,delhi,2024 Kia Sonet HTK Plus,₹8.40 Lakh,0,0,10000,Petrol,Manual,2024,Kia,Sonet,HTK Plus
1,delhi,2022 Renault Kiger RXZ,₹5.75 Lakh,0,0,50214,Petrol,Manual,2022,Renault,Kiger,RXZ
2,delhi,2024 Nissan Magnite XV,₹6.80 Lakh,0,0,19000,Petrol,Manual,2024,Nissan,Magnite,XV
3,delhi,2022 Renault Kiger RXT Opt,₹5.50 Lakh,0,0,14464,Petrol,Manual,2022,Renault,Kiger,RXT Opt
4,delhi,2022 Kia Sonet HTX Turbo iMT BSVI,₹8 Lakh,0,0,37741,Petrol,Manual,2022,Kia,Sonet,HTX Turbo iMT BSVI
...,...,...,...,...,...,...,...,...,...,...,...,...
15595,chandigarh,2023 Maruti Alto K10 VXI,₹3.80 Lakh,₹4.19 Lakh,"(Save ₹39,131)",28199,Petrol,Manual,2023,Maruti,Alto,K10 VXI
15596,chandigarh,2024 MG Astor Select CVT,₹12.50 Lakh,0,0,20000,Petrol,Automatic,2024,MG,Astor,Select CVT
15597,chandigarh,2024 Mahindra XUV 3XO MX3,₹8.50 Lakh,0,0,8000,Petrol,Manual,2024,Mahindra,XUV,3XO MX3
15598,chandigarh,2024 Kia Sonet HTX Turbo iMT,₹10.70 Lakh,0,0,17000,Petrol,Manual,2024,Kia,Sonet,HTX Turbo iMT


In [22]:
new_order = [
    "Year",
    "City",
    "Brand",
    "Model",
    "Variant",
    "Price",
    "Old Price",
    "Savings",
    "Transmission",
    "Fuel_Type",
    "KM_Driven",
    "Title"
]

df = df[new_order]


In [23]:
df

,Year,City,Brand,Model,Variant,Price,Old Price,Savings,Transmission,Fuel_Type,KM_Driven,Title
0,2024,delhi,Kia,Sonet,HTK Plus,₹8.40 Lakh,0,0,Manual,Petrol,10000,2024 Kia Sonet HTK Plus
1,2022,delhi,Renault,Kiger,RXZ,₹5.75 Lakh,0,0,Manual,Petrol,50214,2022 Renault Kiger RXZ
2,2024,delhi,Nissan,Magnite,XV,₹6.80 Lakh,0,0,Manual,Petrol,19000,2024 Nissan Magnite XV
3,2022,delhi,Renault,Kiger,RXT Opt,₹5.50 Lakh,0,0,Manual,Petrol,14464,2022 Renault Kiger RXT Opt
4,2022,delhi,Kia,Sonet,HTX Turbo iMT BSVI,₹8 Lakh,0,0,Manual,Petrol,37741,2022 Kia Sonet HTX Turbo iMT BSVI
...,...,...,...,...,...,...,...,...,...,...,...,...
15595,2023,chandigarh,Maruti,Alto,K10 VXI,₹3.80 Lakh,₹4.19 Lakh,"(Save ₹39,131)",Manual,Petrol,28199,2023 Maruti Alto K10 VXI
15596,2024,chandigarh,MG,Astor,Select CVT,₹12.50 Lakh,0,0,Automatic,Petrol,20000,2024 MG Astor Select CVT
15597,2024,chandigarh,Mahindra,XUV,3XO MX3,₹8.50 Lakh,0,0,Manual,Petrol,8000,2024 Mahindra XUV 3XO MX3
15598,2024,chandigarh,Kia,Sonet,HTX Turbo iMT,₹10.70 Lakh,0,0,Manual,Petrol,17000,2024 Kia Sonet HTX Turbo iMT


In [24]:
df.drop(columns = ["Title"], inplace = True)

C:\Users\vaish\AppData\Local\Temp\ipykernel_17168\51888434.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns = ["Title"], inplace = True)


In [25]:
df

,Year,City,Brand,Model,Variant,Price,Old Price,Savings,Transmission,Fuel_Type,KM_Driven
0,2024,delhi,Kia,Sonet,HTK Plus,₹8.40 Lakh,0,0,Manual,Petrol,10000
1,2022,delhi,Renault,Kiger,RXZ,₹5.75 Lakh,0,0,Manual,Petrol,50214
2,2024,delhi,Nissan,Magnite,XV,₹6.80 Lakh,0,0,Manual,Petrol,19000
3,2022,delhi,Renault,Kiger,RXT Opt,₹5.50 Lakh,0,0,Manual,Petrol,14464
4,2022,delhi,Kia,Sonet,HTX Turbo iMT BSVI,₹8 Lakh,0,0,Manual,Petrol,37741
...,...,...,...,...,...,...,...,...,...,...,...
15595,2023,chandigarh,Maruti,Alto,K10 VXI,₹3.80 Lakh,₹4.19 Lakh,"(Save ₹39,131)",Manual,Petrol,28199
15596,2024,chandigarh,MG,Astor,Select CVT,₹12.50 Lakh,0,0,Automatic,Petrol,20000
15597,2024,chandigarh,Mahindra,XUV,3XO MX3,₹8.50 Lakh,0,0,Manual,Petrol,8000
15598,2024,chandigarh,Kia,Sonet,HTX Turbo iMT,₹10.70 Lakh,0,0,Manual,Petrol,17000


In [26]:
df.to_csv("CarDekho_India_UsedCars_CLEANED.csv", index=False)

In [27]:
df

,Year,City,Brand,Model,Variant,Price,Old Price,Savings,Transmission,Fuel_Type,KM_Driven
0,2024,delhi,Kia,Sonet,HTK Plus,₹8.40 Lakh,0,0,Manual,Petrol,10000
1,2022,delhi,Renault,Kiger,RXZ,₹5.75 Lakh,0,0,Manual,Petrol,50214
2,2024,delhi,Nissan,Magnite,XV,₹6.80 Lakh,0,0,Manual,Petrol,19000
3,2022,delhi,Renault,Kiger,RXT Opt,₹5.50 Lakh,0,0,Manual,Petrol,14464
4,2022,delhi,Kia,Sonet,HTX Turbo iMT BSVI,₹8 Lakh,0,0,Manual,Petrol,37741
...,...,...,...,...,...,...,...,...,...,...,...
15595,2023,chandigarh,Maruti,Alto,K10 VXI,₹3.80 Lakh,₹4.19 Lakh,"(Save ₹39,131)",Manual,Petrol,28199
15596,2024,chandigarh,MG,Astor,Select CVT,₹12.50 Lakh,0,0,Automatic,Petrol,20000
15597,2024,chandigarh,Mahindra,XUV,3XO MX3,₹8.50 Lakh,0,0,Manual,Petrol,8000
15598,2024,chandigarh,Kia,Sonet,HTX Turbo iMT,₹10.70 Lakh,0,0,Manual,Petrol,17000


In [28]:
def convert_price(x):
    if pd.isna(x):
        return 0
    x = str(x).replace("₹","").replace(",","").strip()
    if "Lakh" in x:
        x = x.replace("Lakh","").strip()
        return float(x) * 100000
    return float(x)
    
df["Price"] = df["Price"].apply(convert_price)
df["Old Price"] = df["Old Price"].apply(convert_price)


C:\Users\vaish\AppData\Local\Temp\ipykernel_17168\2017805372.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Price"] = df["Price"].apply(convert_price)
C:\Users\vaish\AppData\Local\Temp\ipykernel_17168\2017805372.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Old Price"] = df["Old Price"].apply(convert_price)


In [29]:
df

,Year,City,Brand,Model,Variant,Price,Old Price,Savings,Transmission,Fuel_Type,KM_Driven
0,2024,delhi,Kia,Sonet,HTK Plus,840000.0,0.0,0,Manual,Petrol,10000
1,2022,delhi,Renault,Kiger,RXZ,575000.0,0.0,0,Manual,Petrol,50214
2,2024,delhi,Nissan,Magnite,XV,680000.0,0.0,0,Manual,Petrol,19000
3,2022,delhi,Renault,Kiger,RXT Opt,550000.0,0.0,0,Manual,Petrol,14464
4,2022,delhi,Kia,Sonet,HTX Turbo iMT BSVI,800000.0,0.0,0,Manual,Petrol,37741
...,...,...,...,...,...,...,...,...,...,...,...
15595,2023,chandigarh,Maruti,Alto,K10 VXI,380000.0,419000.0,"(Save ₹39,131)",Manual,Petrol,28199
15596,2024,chandigarh,MG,Astor,Select CVT,1250000.0,0.0,0,Automatic,Petrol,20000
15597,2024,chandigarh,Mahindra,XUV,3XO MX3,850000.0,0.0,0,Manual,Petrol,8000
15598,2024,chandigarh,Kia,Sonet,HTX Turbo iMT,1070000.0,0.0,0,Manual,Petrol,17000


In [43]:
import re

def convert_savings(x):
    if pd.isna(x):
        return 0
    x = str(x)
    # extract digits only
    num = re.findall(r"\d+", x.replace(",",""))
    if num:
        return int(num[0])
    return 0

df["Savings"] = df["Savings"].apply(convert_savings)


In [44]:
df

,Year,City,Brand,Model,Variant,Price,Old Price,Savings,Transmission,Fuel_Type,KM_Driven
0,2024,delhi,Kia,Sonet,HTK Plus,840000.0,0.0,0,Manual,Petrol,10000
1,2022,delhi,Renault,Kiger,RXZ,575000.0,0.0,0,Manual,Petrol,50214
2,2024,delhi,Nissan,Magnite,XV,680000.0,0.0,0,Manual,Petrol,19000
3,2022,delhi,Renault,Kiger,RXT Opt,550000.0,0.0,0,Manual,Petrol,14464
4,2022,delhi,Kia,Sonet,HTX Turbo iMT BSVI,800000.0,0.0,0,Manual,Petrol,37741
...,...,...,...,...,...,...,...,...,...,...,...
15595,2023,chandigarh,Maruti,Alto,K10 VXI,380000.0,419000.0,39131,Manual,Petrol,28199
15596,2024,chandigarh,MG,Astor,Select CVT,1250000.0,0.0,0,Automatic,Petrol,20000
15597,2024,chandigarh,Mahindra,XUV,3XO MX3,850000.0,0.0,0,Manual,Petrol,8000
15598,2024,chandigarh,Kia,Sonet,HTX Turbo iMT,1070000.0,0.0,0,Manual,Petrol,17000


In [45]:
df[["Price", "Old Price", "Savings"]].dtypes


Price        float64
Old Price    float64
Savings        int64
dtype: object

In [46]:
df["Old Price"] = df["Old Price"].mask(df["Old Price"]==0, df["Price"])

In [3]:
df

NameError: name 'df' is not defined

In [2]:
df.to_csv("CarDekho_India_UsedCars_CLEANED.xlsx", index=False)

NameError: name 'df' is not defined

In [49]:
df

,Year,City,Brand,Model,Variant,Price,Old Price,Savings,Transmission,Fuel_Type,KM_Driven
0,2024,delhi,Kia,Sonet,HTK Plus,840000.0,840000.0,0,Manual,Petrol,10000
1,2022,delhi,Renault,Kiger,RXZ,575000.0,575000.0,0,Manual,Petrol,50214
2,2024,delhi,Nissan,Magnite,XV,680000.0,680000.0,0,Manual,Petrol,19000
3,2022,delhi,Renault,Kiger,RXT Opt,550000.0,550000.0,0,Manual,Petrol,14464
4,2022,delhi,Kia,Sonet,HTX Turbo iMT BSVI,800000.0,800000.0,0,Manual,Petrol,37741
...,...,...,...,...,...,...,...,...,...,...,...
15595,2023,chandigarh,Maruti,Alto,K10 VXI,380000.0,419000.0,39131,Manual,Petrol,28199
15596,2024,chandigarh,MG,Astor,Select CVT,1250000.0,1250000.0,0,Automatic,Petrol,20000
15597,2024,chandigarh,Mahindra,XUV,3XO MX3,850000.0,850000.0,0,Manual,Petrol,8000
15598,2024,chandigarh,Kia,Sonet,HTX Turbo iMT,1070000.0,1070000.0,0,Manual,Petrol,17000


In [50]:
df["Price"] = df["Price"].round(0).astype(int)
df["Old Price"] = df["Old Price"].round(0).astype(int)
df["Savings"] = df["Savings"].astype(int)

In [51]:
df

,Year,City,Brand,Model,Variant,Price,Old Price,Savings,Transmission,Fuel_Type,KM_Driven
0,2024,delhi,Kia,Sonet,HTK Plus,840000,840000,0,Manual,Petrol,10000
1,2022,delhi,Renault,Kiger,RXZ,575000,575000,0,Manual,Petrol,50214
2,2024,delhi,Nissan,Magnite,XV,680000,680000,0,Manual,Petrol,19000
3,2022,delhi,Renault,Kiger,RXT Opt,550000,550000,0,Manual,Petrol,14464
4,2022,delhi,Kia,Sonet,HTX Turbo iMT BSVI,800000,800000,0,Manual,Petrol,37741
...,...,...,...,...,...,...,...,...,...,...,...
15595,2023,chandigarh,Maruti,Alto,K10 VXI,380000,419000,39131,Manual,Petrol,28199
15596,2024,chandigarh,MG,Astor,Select CVT,1250000,1250000,0,Automatic,Petrol,20000
15597,2024,chandigarh,Mahindra,XUV,3XO MX3,850000,850000,0,Manual,Petrol,8000
15598,2024,chandigarh,Kia,Sonet,HTX Turbo iMT,1070000,1070000,0,Manual,Petrol,17000


In [52]:
df.to_csv("CarDekho_India_UsedCars_CLEANED.csv", index=False)

In [53]:
df

,Year,City,Brand,Model,Variant,Price,Old Price,Savings,Transmission,Fuel_Type,KM_Driven
0,2024,delhi,Kia,Sonet,HTK Plus,840000,840000,0,Manual,Petrol,10000
1,2022,delhi,Renault,Kiger,RXZ,575000,575000,0,Manual,Petrol,50214
2,2024,delhi,Nissan,Magnite,XV,680000,680000,0,Manual,Petrol,19000
3,2022,delhi,Renault,Kiger,RXT Opt,550000,550000,0,Manual,Petrol,14464
4,2022,delhi,Kia,Sonet,HTX Turbo iMT BSVI,800000,800000,0,Manual,Petrol,37741
...,...,...,...,...,...,...,...,...,...,...,...
15595,2023,chandigarh,Maruti,Alto,K10 VXI,380000,419000,39131,Manual,Petrol,28199
15596,2024,chandigarh,MG,Astor,Select CVT,1250000,1250000,0,Automatic,Petrol,20000
15597,2024,chandigarh,Mahindra,XUV,3XO MX3,850000,850000,0,Manual,Petrol,8000
15598,2024,chandigarh,Kia,Sonet,HTX Turbo iMT,1070000,1070000,0,Manual,Petrol,17000


In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15600 entries, 0 to 15599
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Year          15600 non-null  int64 
 1   City          15600 non-null  object
 2   Brand         15600 non-null  object
 3   Model         15600 non-null  object
 4   Variant       15600 non-null  object
 5   Price         15600 non-null  int64 
 6   Old Price     15600 non-null  int64 
 7   Savings       15600 non-null  int64 
 8   Transmission  15600 non-null  object
 9   Fuel_Type     15600 non-null  object
 10  KM_Driven     15600 non-null  int64 
dtypes: int64(5), object(6)
memory usage: 1.3+ MB


In [34]:
#convert year -> object 
df["Year"] = df["Year"].astype(str)

C:\Users\vaish\AppData\Local\Temp\ipykernel_17168\3822305198.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Year"] = df["Year"].astype(str)


In [35]:
df

,Year,City,Brand,Model,Variant,Price,Old Price,Savings,Transmission,Fuel_Type,KM_Driven
0,2024,delhi,Kia,Sonet,HTK Plus,840000.0,0.0,0,Manual,Petrol,10000
1,2022,delhi,Renault,Kiger,RXZ,575000.0,0.0,0,Manual,Petrol,50214
2,2024,delhi,Nissan,Magnite,XV,680000.0,0.0,0,Manual,Petrol,19000
3,2022,delhi,Renault,Kiger,RXT Opt,550000.0,0.0,0,Manual,Petrol,14464
4,2022,delhi,Kia,Sonet,HTX Turbo iMT BSVI,800000.0,0.0,0,Manual,Petrol,37741
...,...,...,...,...,...,...,...,...,...,...,...
15595,2023,chandigarh,Maruti,Alto,K10 VXI,380000.0,419000.0,"(Save ₹39,131)",Manual,Petrol,28199
15596,2024,chandigarh,MG,Astor,Select CVT,1250000.0,0.0,0,Automatic,Petrol,20000
15597,2024,chandigarh,Mahindra,XUV,3XO MX3,850000.0,0.0,0,Manual,Petrol,8000
15598,2024,chandigarh,Kia,Sonet,HTX Turbo iMT,1070000.0,0.0,0,Manual,Petrol,17000


In [36]:
import re

def convert_savings(x):
    if pd.isna(x):
        return 0
    x = str(x)
    num = re.findall(r"\d+", x.replace(",", ""))
    if num:
        return int(num[0])
    return 0

df["Savings"] = df["Savings"].apply(convert_savings)
df

C:\Users\vaish\AppData\Local\Temp\ipykernel_17168\3106000835.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Savings"] = df["Savings"].apply(convert_savings)


,Year,City,Brand,Model,Variant,Price,Old Price,Savings,Transmission,Fuel_Type,KM_Driven
0,2024,delhi,Kia,Sonet,HTK Plus,840000.0,0.0,0,Manual,Petrol,10000
1,2022,delhi,Renault,Kiger,RXZ,575000.0,0.0,0,Manual,Petrol,50214
2,2024,delhi,Nissan,Magnite,XV,680000.0,0.0,0,Manual,Petrol,19000
3,2022,delhi,Renault,Kiger,RXT Opt,550000.0,0.0,0,Manual,Petrol,14464
4,2022,delhi,Kia,Sonet,HTX Turbo iMT BSVI,800000.0,0.0,0,Manual,Petrol,37741
...,...,...,...,...,...,...,...,...,...,...,...
15595,2023,chandigarh,Maruti,Alto,K10 VXI,380000.0,419000.0,39131,Manual,Petrol,28199
15596,2024,chandigarh,MG,Astor,Select CVT,1250000.0,0.0,0,Automatic,Petrol,20000
15597,2024,chandigarh,Mahindra,XUV,3XO MX3,850000.0,0.0,0,Manual,Petrol,8000
15598,2024,chandigarh,Kia,Sonet,HTX Turbo iMT,1070000.0,0.0,0,Manual,Petrol,17000


In [37]:
df["Old Price"] = df["Old Price"].mask(df["Old Price"] == 0, df["Price"])
df

C:\Users\vaish\AppData\Local\Temp\ipykernel_17168\2436154690.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Old Price"] = df["Old Price"].mask(df["Old Price"] == 0, df["Price"])


,Year,City,Brand,Model,Variant,Price,Old Price,Savings,Transmission,Fuel_Type,KM_Driven
0,2024,delhi,Kia,Sonet,HTK Plus,840000.0,840000.0,0,Manual,Petrol,10000
1,2022,delhi,Renault,Kiger,RXZ,575000.0,575000.0,0,Manual,Petrol,50214
2,2024,delhi,Nissan,Magnite,XV,680000.0,680000.0,0,Manual,Petrol,19000
3,2022,delhi,Renault,Kiger,RXT Opt,550000.0,550000.0,0,Manual,Petrol,14464
4,2022,delhi,Kia,Sonet,HTX Turbo iMT BSVI,800000.0,800000.0,0,Manual,Petrol,37741
...,...,...,...,...,...,...,...,...,...,...,...
15595,2023,chandigarh,Maruti,Alto,K10 VXI,380000.0,419000.0,39131,Manual,Petrol,28199
15596,2024,chandigarh,MG,Astor,Select CVT,1250000.0,1250000.0,0,Automatic,Petrol,20000
15597,2024,chandigarh,Mahindra,XUV,3XO MX3,850000.0,850000.0,0,Manual,Petrol,8000
15598,2024,chandigarh,Kia,Sonet,HTX Turbo iMT,1070000.0,1070000.0,0,Manual,Petrol,17000


In [40]:
df["Price"] = df["Price"].round(0).astype(int)
df

C:\Users\vaish\AppData\Local\Temp\ipykernel_17168\4157958988.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Price"] = df["Price"].round(0).astype(int)


,Year,City,Brand,Model,Variant,Price,Old Price,Savings,Transmission,Fuel_Type,KM_Driven
0,2024,delhi,Kia,Sonet,HTK Plus,840000,840000.0,0,Manual,Petrol,10000
1,2022,delhi,Renault,Kiger,RXZ,575000,575000.0,0,Manual,Petrol,50214
2,2024,delhi,Nissan,Magnite,XV,680000,680000.0,0,Manual,Petrol,19000
3,2022,delhi,Renault,Kiger,RXT Opt,550000,550000.0,0,Manual,Petrol,14464
4,2022,delhi,Kia,Sonet,HTX Turbo iMT BSVI,800000,800000.0,0,Manual,Petrol,37741
...,...,...,...,...,...,...,...,...,...,...,...
15595,2023,chandigarh,Maruti,Alto,K10 VXI,380000,419000.0,39131,Manual,Petrol,28199
15596,2024,chandigarh,MG,Astor,Select CVT,1250000,1250000.0,0,Automatic,Petrol,20000
15597,2024,chandigarh,Mahindra,XUV,3XO MX3,850000,850000.0,0,Manual,Petrol,8000
15598,2024,chandigarh,Kia,Sonet,HTX Turbo iMT,1070000,1070000.0,0,Manual,Petrol,17000


In [41]:
df["Old Price"] = df["Old Price"].round(0).astype(int)
df

C:\Users\vaish\AppData\Local\Temp\ipykernel_17168\1260702932.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Old Price"] = df["Old Price"].round(0).astype(int)


,Year,City,Brand,Model,Variant,Price,Old Price,Savings,Transmission,Fuel_Type,KM_Driven
0,2024,delhi,Kia,Sonet,HTK Plus,840000,840000,0,Manual,Petrol,10000
1,2022,delhi,Renault,Kiger,RXZ,575000,575000,0,Manual,Petrol,50214
2,2024,delhi,Nissan,Magnite,XV,680000,680000,0,Manual,Petrol,19000
3,2022,delhi,Renault,Kiger,RXT Opt,550000,550000,0,Manual,Petrol,14464
4,2022,delhi,Kia,Sonet,HTX Turbo iMT BSVI,800000,800000,0,Manual,Petrol,37741
...,...,...,...,...,...,...,...,...,...,...,...
15595,2023,chandigarh,Maruti,Alto,K10 VXI,380000,419000,39131,Manual,Petrol,28199
15596,2024,chandigarh,MG,Astor,Select CVT,1250000,1250000,0,Automatic,Petrol,20000
15597,2024,chandigarh,Mahindra,XUV,3XO MX3,850000,850000,0,Manual,Petrol,8000
15598,2024,chandigarh,Kia,Sonet,HTX Turbo iMT,1070000,1070000,0,Manual,Petrol,17000


In [43]:
df.dtypes

Year            object
City            object
Brand           object
Model           object
Variant         object
Price            int64
Old Price        int64
Savings          int64
Transmission    object
Fuel_Type       object
KM_Driven        int64
dtype: object

In [44]:
num= df.select_dtypes(include= 'int64')
print('Numerical_df', num.columns)
print('*'*60)
cat= df.select_dtypes(include= 'object')
print('CAtegorical_df' , cat.columns)

Numerical_df Index(['Price', 'Old Price', 'Savings', 'KM_Driven'], dtype='object')
************************************************************
CAtegorical_df Index(['Year', 'City', 'Brand', 'Model', 'Variant', 'Transmission',
       'Fuel_Type'],
      dtype='object')


In [45]:
num

,Price,Old Price,Savings,KM_Driven
0,840000,840000,0,10000
1,575000,575000,0,50214
2,680000,680000,0,19000
3,550000,550000,0,14464
4,800000,800000,0,37741
...,...,...,...,...
15595,380000,419000,39131,28199
15596,1250000,1250000,0,20000
15597,850000,850000,0,8000
15598,1070000,1070000,0,17000


In [46]:
df.dtypes

Year            object
City            object
Brand           object
Model           object
Variant         object
Price            int64
Old Price        int64
Savings          int64
Transmission    object
Fuel_Type       object
KM_Driven        int64
dtype: object

In [1]:
from IPython.display import FileLink
FileLink('CarDekho_India_UsedCars_CLEANED.csv')


C:\Users\vaish\CarData\CarDekho_India_UsedCars_CLEANED.csv

In [47]:
df.to_csv("CarDekho_India_UsedCars_CLEANED.csv", index=False)

In [48]:
df

,Year,City,Brand,Model,Variant,Price,Old Price,Savings,Transmission,Fuel_Type,KM_Driven
0,2024,delhi,Kia,Sonet,HTK Plus,840000,840000,0,Manual,Petrol,10000
1,2022,delhi,Renault,Kiger,RXZ,575000,575000,0,Manual,Petrol,50214
2,2024,delhi,Nissan,Magnite,XV,680000,680000,0,Manual,Petrol,19000
3,2022,delhi,Renault,Kiger,RXT Opt,550000,550000,0,Manual,Petrol,14464
4,2022,delhi,Kia,Sonet,HTX Turbo iMT BSVI,800000,800000,0,Manual,Petrol,37741
...,...,...,...,...,...,...,...,...,...,...,...
15595,2023,chandigarh,Maruti,Alto,K10 VXI,380000,419000,39131,Manual,Petrol,28199
15596,2024,chandigarh,MG,Astor,Select CVT,1250000,1250000,0,Automatic,Petrol,20000
15597,2024,chandigarh,Mahindra,XUV,3XO MX3,850000,850000,0,Manual,Petrol,8000
15598,2024,chandigarh,Kia,Sonet,HTX Turbo iMT,1070000,1070000,0,Manual,Petrol,17000


In [49]:
from google.colab import files
files.download("cleaned_cars.csv")


ModuleNotFoundError: No module named 'google.colab'

In [51]:
import base64
from IPython.display import HTML

filename = "CarDekho_India_UsedCars_CLEANED.csv"

def create_download_link(filename):
    with open(filename, "rb") as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    return HTML(f'<a download="{filename}" href="data:text/csv;base64,{b64}">📥 Click Here to Download CarDekho_India_UsedCars_CLEANED.csv</a>')

create_download_link(filename)


In [2]:
import pandas as pd
df = pd.read_csv("CarDekho_India_UsedCars_CLEANED.csv")
df

,Year,City,Brand,Model,Variant,Price,Old Price,Savings,Transmission,Fuel_Type,KM_Driven
0,2024,delhi,Kia,Sonet,HTK Plus,840000,840000,0,Manual,Petrol,10000
1,2022,delhi,Renault,Kiger,RXZ,575000,575000,0,Manual,Petrol,50214
2,2024,delhi,Nissan,Magnite,XV,680000,680000,0,Manual,Petrol,19000
3,2022,delhi,Renault,Kiger,RXT Opt,550000,550000,0,Manual,Petrol,14464
4,2022,delhi,Kia,Sonet,HTX Turbo iMT BSVI,800000,800000,0,Manual,Petrol,37741
...,...,...,...,...,...,...,...,...,...,...,...
15595,2023,chandigarh,Maruti,Alto,K10 VXI,380000,419000,39131,Manual,Petrol,28199
15596,2024,chandigarh,MG,Astor,Select CVT,1250000,1250000,0,Automatic,Petrol,20000
15597,2024,chandigarh,Mahindra,XUV,3XO MX3,850000,850000,0,Manual,Petrol,8000
15598,2024,chandigarh,Kia,Sonet,HTX Turbo iMT,1070000,1070000,0,Manual,Petrol,17000


In [3]:
def clean_price(x):
    if isinstance(x, str):
        x = x.replace("₹","").replace(",","").strip()
        if "Lakh" in x:
            return float(x.replace("Lakh","")) * 100000
        elif x == "0" or x == "":
            return 0
    return float(x)

df["Price"] = df["Price"].apply(clean_price)


In [4]:
df

,Year,City,Brand,Model,Variant,Price,Old Price,Savings,Transmission,Fuel_Type,KM_Driven
0,2024,delhi,Kia,Sonet,HTK Plus,840000.0,840000,0,Manual,Petrol,10000
1,2022,delhi,Renault,Kiger,RXZ,575000.0,575000,0,Manual,Petrol,50214
2,2024,delhi,Nissan,Magnite,XV,680000.0,680000,0,Manual,Petrol,19000
3,2022,delhi,Renault,Kiger,RXT Opt,550000.0,550000,0,Manual,Petrol,14464
4,2022,delhi,Kia,Sonet,HTX Turbo iMT BSVI,800000.0,800000,0,Manual,Petrol,37741
...,...,...,...,...,...,...,...,...,...,...,...
15595,2023,chandigarh,Maruti,Alto,K10 VXI,380000.0,419000,39131,Manual,Petrol,28199
15596,2024,chandigarh,MG,Astor,Select CVT,1250000.0,1250000,0,Automatic,Petrol,20000
15597,2024,chandigarh,Mahindra,XUV,3XO MX3,850000.0,850000,0,Manual,Petrol,8000
15598,2024,chandigarh,Kia,Sonet,HTX Turbo iMT,1070000.0,1070000,0,Manual,Petrol,17000


In [5]:
df["Old Price"] = df["Old Price"].replace(0, df["Price"])


ValueError: Series.replace cannot use dict-value and non-None to_replace

In [6]:
df

,Year,City,Brand,Model,Variant,Price,Old Price,Savings,Transmission,Fuel_Type,KM_Driven
0,2024,delhi,Kia,Sonet,HTK Plus,840000.0,840000,0,Manual,Petrol,10000
1,2022,delhi,Renault,Kiger,RXZ,575000.0,575000,0,Manual,Petrol,50214
2,2024,delhi,Nissan,Magnite,XV,680000.0,680000,0,Manual,Petrol,19000
3,2022,delhi,Renault,Kiger,RXT Opt,550000.0,550000,0,Manual,Petrol,14464
4,2022,delhi,Kia,Sonet,HTX Turbo iMT BSVI,800000.0,800000,0,Manual,Petrol,37741
...,...,...,...,...,...,...,...,...,...,...,...
15595,2023,chandigarh,Maruti,Alto,K10 VXI,380000.0,419000,39131,Manual,Petrol,28199
15596,2024,chandigarh,MG,Astor,Select CVT,1250000.0,1250000,0,Automatic,Petrol,20000
15597,2024,chandigarh,Mahindra,XUV,3XO MX3,850000.0,850000,0,Manual,Petrol,8000
15598,2024,chandigarh,Kia,Sonet,HTX Turbo iMT,1070000.0,1070000,0,Manual,Petrol,17000
